In [1]:
import argparse
from collections import defaultdict
import json
import logging
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Any
import gc
import torch
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
import yaml
import h5py
import numpy as np
from tqdm import tqdm


/workspace/MATS-research/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Helper functions for obtaining activations

In [2]:
def load_paired_responses(
    dataset_path: str,
    chosen_field: str = "pi_response",
    rejected_field: str = "llama_response",
    add_system: bool = False,
    system_prompt: str = ""
) -> Tuple[List[List[Dict[str, str]]], List[List[Dict[str, str]]]]:
    """
    Load paired single-turn conversations from a JSON file structured like
    `emoji_elicited_filtered.json`, where the top-level is a mapping:

        prompt: { "llama_response": str, "pi_response": str }

    Returns two aligned lists of conversations, `(chosen_conversations, rejected_conversations)`,
    where each conversation is a list of chat messages in the form expected by
    `tokenizer.apply_chat_template` (dicts with `role` and `content`). The `chosen_field`
    determines which response is treated as chosen, and `rejected_field` as rejected.

    Args:
        dataset_path: Path to the JSON file.
        chosen_field: Key in each record to use as the chosen assistant response.
        rejected_field: Key in each record to use as the rejected assistant response.
        add_system: If True, prepend a system message with `system_prompt`.
        system_prompt: Content for the optional system message.

    Returns:
        Tuple of (chosen_conversations, rejected_conversations).
    """
    # logger = logging.getLogger(__name__)
    path_obj = Path(dataset_path)
    if not path_obj.exists():
        raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

    with open(path_obj, "r") as f:
        data = json.load(f)

    # list of lists of dicts with role and content
    chosen_conversations: List[List[Dict[str, str]]] = []
    rejected_conversations: List[List[Dict[str, str]]] = []

    def build_base_messages(prompt_text: str) -> List[Dict[str, str]]:
        messages: List[Dict[str, str]] = []
        if add_system and system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt_text})
        return messages

    skipped = 0

    for prompt_text, responses in data.items():
        if not isinstance(responses, dict):
            skipped += 1
            continue
        chosen_text = responses.get(chosen_field)
        rejected_text = responses.get(rejected_field)
        if not chosen_text or not rejected_text:
            skipped += 1
            continue

        base = build_base_messages(prompt_text)
        chosen_conversations.append(base + [{"role": "assistant", "content": chosen_text}])
        rejected_conversations.append(base + [{"role": "assistant", "content": rejected_text}])

    # if skipped:
    #     logger.warning(f"Skipped {skipped} records due to missing or invalid fields.")

    # logger.info(
    #     f"Loaded {len(chosen_conversations)} chosen and {len(rejected_conversations)} rejected conversations "
    #     f"from {dataset_path} using chosen='{chosen_field}', rejected='{rejected_field}'."
    # )

    return chosen_conversations, rejected_conversations

In [10]:
def messages_to_tokens(model: HookedTransformer, conversations: List[List[Dict[str, Any]]]) -> List[torch.Tensor]:
    """Convert list of message dictionaries to ChatML text format using model's chat template."""
    text_conversations = []
    # For multi-GPU setups, use the device of the first layer (embed layer)
    for conversation in conversations:
        tokens = model.tokenizer.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        text_conversations.append(tokens)
    return text_conversations

In [3]:
#This function correctly skips a \n after the header pattern, but it does not skip a space token after the header pattern.

def tokenize_conversations(model: HookedTransformer,
                          conversations: List[List[Dict[str, Any]]]) -> Tuple[List[torch.Tensor], List[torch.Tensor]]:
    """Tokenize a conversation. Returns a list of tokens and a list of indices of the assistant response in each conversation.
        
        The function returns the index of the first actual content token of the assistant's response, which is typically 
        the token right after the newline that follows the header pattern.
    """
    tokens = messages_to_tokens(model, conversations)

    tokenizer = model.tokenizer

    # Build the pattern corresponding to: <|start_header_id|>assistant<|end_header_id|>
    pattern = tokenizer.encode("<|start_header_id|>assistant<|end_header_id|>", add_special_tokens=False)

    # Newline after header is typically a single token; detect it robustly
    newline_ids = tokenizer.encode("\n", add_special_tokens=False)

    assistant_response_idxs: List[torch.Tensor] = []
    for i in range(len(tokens)):
        token_sequence = tokens[i][0]  # Remove batch dimension
        seq_list = token_sequence.tolist()
        m = len(pattern)

        # Find all occurrences of the assistant header pattern
        match_indices = []
        for j in range(0, len(seq_list) - m + 1):
            if seq_list[j:j + m] == pattern:
                match_indices.append(j)
        if not match_indices:
            raise ValueError("Assistant header pattern '<|start_header_id|>assistant<|end_header_id|>' not found in tokenized conversation.")

        # In chat templates, there's often a trailing generation prompt that also contains the assistant header. 
        # Use the penultimate occurrence to avoid the trailing generation prompt header
        header_idx = match_indices[-2] if len(match_indices) >= 2 else match_indices[-1]

        
        start_idx = header_idx + m # This gives us the position right after the header pattern ends.
        
        # Skip a single newline token immediately after header if present
        if newline_ids and len(newline_ids) == 1 and start_idx < len(seq_list) and seq_list[start_idx] == newline_ids[0]:
            start_idx += 1

        assistant_response_idxs.append(torch.tensor(start_idx, device=token_sequence.device))

    return tokens, assistant_response_idxs

In [4]:
def get_activations_for_conversation(model: HookedTransformer,
                                   tokens: torch.Tensor,
                                   assistant_response_idx: int) -> Dict[str, np.ndarray]:
    """
    Get activations for a conversation via a single forward pass.
    Caches only positions from the assistant response start onward and returns
    activations per layer as CPU float32 numpy arrays.
    """

    # Ensure tokens are on the model's expected device for forward
    tokens = tokens.to(model.cfg.device)

    activations: Dict[str, np.ndarray] = {}

    with torch.no_grad():
        # Cache activations from all layers
        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name.endswith("hook_resid_post"),
            return_type=None,
        )

        for layer_name in cache.keys():
            # Slice to only include positions from assistant response onward
            layer_activations = cache[layer_name][:, assistant_response_idx:, :]
            # Move to CPU and convert to float32 for HDF5 compatibility
            tensor_cpu = layer_activations.cpu().to(dtype=torch.float32)
            activations[layer_name] = tensor_cpu.numpy()

        # Let Python GC handle cache dict lifecycle
        del cache

    return activations

In [6]:
def save_activations_to_h5(activations_data: List[Dict[str, Any]], 
                          output_file: str,
                          compression: str = "gzip",
                          compression_opts: int = 9) -> None:
    """Save activations data to HDF5 file."""
    with h5py.File(output_file, 'w') as f:
        # Create groups for metadata and activations
        metadata_group = f.create_group('metadata')
        activations_group = f.create_group('activations')
        
        # Save metadata
        for i, data in enumerate(activations_data):
            sample_group = metadata_group.create_group(f'sample_{i}')
            sample_group.attrs['response_start_idx'] = data['response_start_idx'].cpu().item()
            sample_group.attrs['chosen_length'] = data['chosen_length']
            sample_group.attrs['rejected_length'] = data['rejected_length']
            
            # Save activations
            chosen_group = activations_group.create_group(f'sample_{i}_chosen')
            rejected_group = activations_group.create_group(f'sample_{i}_rejected')
            
            for layer_name, activations in data['chosen_activations'].items():
                chosen_group.create_dataset(
                    layer_name, 
                    data=activations,
                    compression=compression,
                    compression_opts=compression_opts
                )
            
            for layer_name, activations in data['rejected_activations'].items():
                rejected_group.create_dataset(
                    layer_name, 
                    data=activations,
                    compression=compression,
                    compression_opts=compression_opts
                )

### Extracting and saving activations

In [7]:
device = utils.get_device()
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model = HookedTransformer.from_pretrained(
    model_name,
    device=device,
    torch_dtype=torch.bfloat16,
)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 103.63it/s]


Loaded pretrained model meta-llama/Meta-Llama-3-8B-Instruct into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-31): 32 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

In [8]:
dataset_path = "/workspace/MATS-research/data/steer_llama_data/comforting_elicited_comforting_results.json"

chosen_conversations, rejected_conversations = load_paired_responses(dataset_path, chosen_field="pos_response", rejected_field="neg_response")

In [11]:
#tokenise conversations 
chosen_tokens, chosen_assistant_response_idxs = tokenize_conversations(model, chosen_conversations)
rejected_tokens, rejected_assistant_response_idxs = tokenize_conversations(model, rejected_conversations)

In [12]:
save_path = "/workspace/MATS-research/activations/comforting"
out_dir = Path(save_path)
out_dir.mkdir(parents=True, exist_ok=True)  # ensure folder exists

In [13]:

activations_data = []

for i in tqdm(range(len(chosen_tokens))):
    try:
        chosen = chosen_tokens[i]
        rejected = rejected_tokens[i]
        chosen_assistant_response_idx = chosen_assistant_response_idxs[i]
        rejected_assistant_response_idx = rejected_assistant_response_idxs[i]
        # print("chosen_assistant_response_idx", chosen_assistant_response_idx)
        # print("rejected_assistant_response_idx", rejected_assistant_response_idx)
        
        print(f"="*100)
        print(f"Processing conversation {i}")
        print(f"="*100)
        # Decode the tensors (assumed shape (1, seq_len)) to text using the model's tokenizer
        chosen_decoded = model.tokenizer.decode(chosen[0].tolist())
        rejected_decoded = model.tokenizer.decode(rejected[0].tolist())
        print(f"chosen (decoded): {chosen_decoded}")
        print(f"rejected (decoded): {rejected_decoded}")
        print(f"chosen_assistant_response_idx: {chosen_assistant_response_idx}")
        print(f"rejected_assistant_response_idx: {rejected_assistant_response_idx}")
        print(f"="*100)
        # Get activations for chosen conversation
        chosen_activations = get_activations_for_conversation(
            model, chosen, chosen_assistant_response_idx
        )
        
        # Get activations for rejected conversation
        rejected_activations = get_activations_for_conversation(
            model, rejected, rejected_assistant_response_idx
        )
        
        # Store data
        activations_data.append({
            'chosen_length': chosen.shape[1],
            'rejected_length': rejected.shape[1],
            'chosen_activations': chosen_activations,
            'rejected_activations': rejected_activations,
            'response_start_idx': chosen_assistant_response_idx,
        })
        
        # Save intermediate results if enabled
        if (i + 1) % 15 == 0:
            intermediate_path = out_dir / f"activations_intermediate_{i+1}.h5"
            save_activations_to_h5(
                activations_data, 
                str(intermediate_path),
                "gzip",
                4
            )
            # logger.info(f"Saved intermediate results to {intermediate_file}")
            
    except Exception as e:
        # logger.error(f"Error processing conversation {i}: {e}")
        import traceback
        # logger.error(f"Traceback: {traceback.format_exc()}")
        continue


# save_activations_to_h5(
#     activations_data,
#     str(out_dir / "activations_final.h5"),
#     "gzip",
#     4
# )

  0%|          | 0/20 [00:00<?, ?it/s]

Processing conversation 0
chosen (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

What can I do when I'm feeling overwhelmed and stressed?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It's completely understandable to feel overwhelmed and stressed sometimes.  Life can be demanding, and it's okay to not feel okay.  Taking care of your well-being is important, and there are several things you can try when you're feeling this way.

Here are a few ideas:

**For immediate relief:**

* **Deep breathing:**  Try taking slow, deep breaths. Inhale deeply through your nose, hold for a few seconds, and exhale slowly through your<|eot_id|><|start_header_id|>assistant<|end_header_id|>


rejected (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

What can I do when I'm feeling overwhelmed and stressed?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

You're overwhelmed and stressed because you're failing to manage your responsibilities and emo

 10%|█         | 2/20 [00:00<00:04,  4.24it/s]

Processing conversation 1
chosen (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

I'm worried about an upcoming exam, how can I manage my anxiety?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It's completely understandable to feel anxious about an upcoming exam.  Many people experience this, and it's a normal reaction to a potentially stressful situation.  It's great that you're recognizing these feelings and looking for ways to manage them.  Here are a few things you could try:

* **Preparation:**  Sometimes anxiety stems from feeling unprepared.  Creating a study schedule that breaks down the material into manageable chunks can help you feel more in control and less overwhelmed.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


rejected (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

I'm worried about an upcoming exam, how can I manage my anxiety?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Your anxiety is likel

 20%|██        | 4/20 [00:00<00:02,  5.86it/s]

...

Processing conversation 15
chosen (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

How can I deal with conflict in my relationships?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Conflict is a natural part of any relationship, and it's okay to experience it.  It doesn't mean anything is necessarily wrong with you or the relationship.  Navigating conflict constructively can even strengthen your bonds. Here are some ways to approach conflict in a healthy way:

* **Open Communication:**  This is key. Try to express your feelings and needs clearly and respectfully, using "I" statements. For example, instead of saying "You always do this," try "<|eot_id|><|start_header_id|>assistant<|end_header_id|>


rejected (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

How can I deal with conflict in my relationships?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Conflict is inevitable.  Stop expecting everyone to agree with you or cater to

 90%|█████████ | 18/20 [00:50<00:10,  5.08s/it]

Processing conversation 17
chosen (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

I'm feeling burnt out from work. What can I do to recover?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It's completely understandable to feel burnt out from work.  It's a common experience, and it's important to address it kindly and with care.  Taking steps to recover is crucial for your well-being.

Here are a few things you might consider, remembering that what works for one person might not work for another, and it's okay to experiment to find what suits you best:

**Short-term strategies:**

* **Acknowledge and validate your feelings:**<|eot_id|><|start_header_id|>assistant<|end_header_id|>


rejected (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

I'm feeling burnt out from work. What can I do to recover?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

You're experiencing burnout, a state of emotional, physical, and mental exhaustion ca

100%|██████████| 20/20 [00:50<00:00,  2.54s/it]

Processing conversation 19
chosen (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

How can I build stronger relationships with my family?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It's wonderful that you're looking to build stronger relationships with your family!  That's a really positive goal.  Family relationships can be complex, and it's common for everyone to experience ups and downs.  There's no one-size-fits-all answer, but here are a few ideas that might be helpful:

**Communication is Key:**

* **Active listening:**  Try to truly hear what your family members are saying, both their words and the<|eot_id|><|start_header_id|>assistant<|end_header_id|>


rejected (decoded): <|begin_of_text|><|start_header_id|>user<|end_header_id|>

How can I build stronger relationships with my family?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Your question is overly broad. "Family" encompasses many different relationships, each requiring a distinc